# 섹션3-12. AI를 활용한 음파 데이터 시각화 - Short term Fourier Transform & Spectrogram

> 강의: [32가지 데이터 시각화 전략 - 비전공자를 위한 기초이론 & 실습](https://www.inflearn.com/course/32-data-visualizatio/dashboard?cid=343563) (반병현) — 전체 23강

- [x] 강의 시청 완료
- [x] 실습/정리 완료

## 배운 내용

<!-- 강의를 보면서 핵심을 적는다 -->

-

## 목표 / 재현할 것

<!-- 이 강의에서 만든 차트를 내 방식대로 다시 만들어본다 -->

-


## 실습

> **데이터:** 개인 음성 녹음 대신, 포트폴리오 케이스
> ([STFT 스펙트로그램](../docs/cases/stft-spectrogram/))의 "데모 음원 생성" 버튼이
> 만드는 것과 같은 합성 신호를 그대로 재현한다 — 목소리는 개인정보라 노트북에 남기지 않는다.
> 신호 구성: 440Hz 순음 + 880Hz 하모닉 + 300→4000Hz 선형 칩(chirp) + 3.2~3.8초 구간의
> 2500Hz 버스트.

In [ ]:
import sys

sys.path.append("..")

import numpy as np
import matplotlib.pyplot as plt

from viz_utils import setup

setup()

sr = 44100  # 표본화 주파수(Hz) — 1초에 44,100번 샘플링
duration = 5.0
t = np.arange(int(sr * duration)) / sr

signal = 0.3 * np.sin(2 * np.pi * 440 * t)          # 기본음 440Hz (A4)
signal += 0.15 * np.sin(2 * np.pi * 880 * t)        # 하모닉 880Hz

chirp_freq = 300 + (4000 - 300) * (t / duration)     # 300Hz -> 4000Hz 선형 칩
signal += 0.25 * np.sin(2 * np.pi * chirp_freq * t)

burst = (t > 3.2) & (t < 3.8)
signal[burst] += 0.2 * np.sin(2 * np.pi * 2500 * t[burst])

print(f"표본 수 {len(signal):,}, {duration}초, {sr:,}Hz")

### 1. 파형만 보면 — 어떤 주파수가 섞여 있는지 안 보인다

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t, signal, lw=0.4, color="#4C78A8")
ax.set(xlabel="시간(초)", ylabel="진폭", title="시간 영역 파형 — 톤/하모닉/칩/버스트가 뒤섞여 안 보인다")
plt.show()

### 2. STFT — 짧은 구간으로 잘라 각 구간에 푸리에 변환

강의 설명 그대로: "쇼트텀 프리 트랜스폼은 짧은 시간 간격에 대해서 프리(푸리에) 트랜스폼을
실행한 것"이다. 전체 5초를 한 번에 변환하면 "언제" 그 주파수가 나타났는지 정보가 사라지므로,
겹치는 짧은 창(window)으로 잘라 각 창마다 주파수 성분을 구하고 시간순으로 이어 붙인다.

In [ ]:
def stft(x, fft_size=1024, hop=256):
    """짧은 창(hann)으로 겹쳐 잘라가며(hop) 각 구간의 주파수 성분을 구한다."""
    window = np.hanning(fft_size)
    n_frames = 1 + (len(x) - fft_size) // hop
    spec = np.empty((fft_size // 2 + 1, n_frames))
    for i in range(n_frames):
        seg = x[i * hop: i * hop + fft_size] * window
        mag = np.abs(np.fft.rfft(seg))
        spec[:, i] = 20 * np.log10(mag + 1e-8)  # dB 스케일
    freqs = np.fft.rfftfreq(fft_size, d=1 / sr)
    times = (np.arange(n_frames) * hop + fft_size / 2) / sr
    return freqs, times, spec


freqs, times, spec = stft(signal, fft_size=1024, hop=256)
print(f"주파수 bin {len(freqs)}개 (0~{freqs[-1]/1000:.1f}kHz), 시간 프레임 {len(times)}개")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
im = ax.pcolormesh(times, freqs, spec, shading="auto", cmap="magma", vmin=-80, vmax=-10)
ax.set(xlabel="시간(초)", ylabel="주파수(Hz)", ylim=(0, 5000),
       title="스펙트로그램 — 이제 4개 성분이 전부 보인다")
fig.colorbar(im, ax=ax, label="진폭(dB)")
plt.show()

> 440Hz·880Hz는 시간축과 평행한 두 개의 수평선으로, 칩은 300Hz에서 4000Hz로
> 대각선을 그리며 올라가는 선으로, 버스트는 3.2~3.8초 구간에 2500Hz 근처에서만 짧게
> 나타나는 점으로 보인다. **파형(1번 그림)에서는 이 4가지가 뭉쳐서 안 보였다.**

### 3. 창 크기(FFT size) — 시간 해상도 vs 주파수 해상도

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, fft_size in zip(axes, [256, 1024, 4096]):
    f, tt, s = stft(signal, fft_size=fft_size, hop=fft_size // 4)
    ax.pcolormesh(tt, f, s, shading="auto", cmap="magma", vmin=-80, vmax=-10)
    ax.set(ylim=(0, 5000), title=f"FFT size={fft_size}", xlabel="시간(초)")
axes[0].set_ylabel("주파수(Hz)")
fig.suptitle("창을 좁히면(256) 버스트 시작·끝이 또렷해지고, 넓히면(4096) 주파수가 촘촘해진다")
plt.tight_layout()
plt.show()

> 창을 좁히면(256) **언제** 일어났는지가 정확해지고, 넓히면(4096) **어떤 주파수**인지가
> 정확해진다 — 동시에 둘 다 정밀할 수 없다(불확정성 원리). 강의에서 "조금 더 정밀 주파수로
> 가볼까요"라며 설정을 바꿔본 것이 바로 이 트레이드오프다.

### 정리 — STFT의 쓰임새 (강의 설명)

전사에 나온 활용처 그대로:

1. **포먼트 분석/모음 분석** — 사람 목소리에서 모음마다 특정 주파수 대역(포먼트)이
   강하게 나타나는 걸 스펙트로그램으로 구분
2. **발성 치료** — 위와 같은 원리로 흉성/두성, 성문 접촉률 등을 시각적으로 추적
3. **노이즈 필터링** — 복잡한 신호에 섞인 특정 주파수 성분(노이즈)만 스펙트로그램에서
   찾아 제거

셋 다 공통점은 "시간에 따라 주파수 구성이 어떻게 바뀌는가"를 봐야 한다는 것이고,
그게 파형 하나만 봐서는 안 되고 STFT가 필요한 이유다.


---

## 메모

-
